<a href="https://colab.research.google.com/github/dishidhak/brown_dhakshin_demand_estimation/blob/main/demand_estimation_air_fryers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand Estimation and Market Analysis: Air Fryers
### Dishitha Dhakshin (qxk8wp), Emerson Brown (gsz5cv)
## 2. Demand Estimation

Welcome!

In [44]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [45]:
df = pd.read_csv("air_fryers_clean_brand_year.csv")

In [46]:
feature_cols = [
    'compact_share', 'dual_basket_share', 'oven_style_share',
    'rotisserie_share', 'window_share'
]

y = df['log_brand_share']

brand_dummies = pd.get_dummies(df['brand'],
                               prefix='brand', drop_first=True, dtype=int)
year_dummies = pd.get_dummies(df['year'].astype(str),
                              prefix='year', drop_first=True, dtype=int)

X = pd.concat(
    [df[['avg_price', 'avg_rating'] + feature_cols],
     brand_dummies,
     year_dummies],
    axis=1
)

model = LinearRegression()
model.fit(X, y)

predicted_log_share = model.predict(X)
r2 = r2_score(y, predicted_log_share)

coef_table = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_
})

print('R-squared:', r2)
coef_table

R-squared: 0.763453950091436


,feature,coefficient
0,avg_price,-0.037668
1,avg_rating,0.287517
2,compact_share,9.815304
3,dual_basket_share,-9.509686
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
6,window_share,12.880298
7,brand_cosori,2.551946
8,brand_cuisinart,6.422436
9,brand_dash,0.176655


In [47]:
price_coef = coef_table.loc[coef_table['feature'] == 'avg_price', 'coefficient'].iloc[0]
print('Estimated price coefficient:', price_coef)

Estimated price coefficient: -0.03766765298429383


The estimated price coefficient of -0.038 is negative. This means higher prices are associated with lower market share. This is consistent with the intuition that as prices increase, product demand decreases. If the coefficient were positive, it would imply that higher prices increase demand, which is unrealistic for air fryers and would make the model unreliable.

In [48]:
rating_coef = coef_table.loc[coef_table['feature'] == 'avg_rating', 'coefficient'].iloc[0]
print('Estimated rating coefficient:', rating_coef)

Estimated rating coefficient: 0.287516847432701


The estimated rating coefficient is positive, meaning higher-rated brands tend to have larger market shares, after controlling for price and product features.

In [49]:
coef_table[coef_table['feature'].isin(feature_cols)].sort_values('coefficient', ascending=False)

,feature,coefficient
6,window_share,12.880298
2,compact_share,9.815304
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
3,dual_basket_share,-9.509686


Dual-basket, compact, and rotisserie features have positive coefficients, which means they are associated with higher demand. In contrast, oven-style and window features have negative coefficients, indicating lower demand. We can observe that consumers tend to prefer more compact and multifunctional designs.  

In [50]:
coef_table[coef_table['feature'].str.startswith('brand_')].sort_values('coefficient', ascending=False)

,feature,coefficient
8,brand_cuisinart,6.422436
12,brand_ninja,5.838705
11,brand_instant_pot,4.626260
10,brand_gowise usa,3.938996
14,brand_oster,3.928074
13,brand_nuwave,3.544883
7,brand_cosori,2.551946
15,brand_ultrean,0.942399
9,brand_dash,0.176655


The brands with the largest positive dummy coefficients are Cuisinart (6.42) and Ninja (5.84), followed by Instant Pot, GoWISE USA, and Oster.

Relative to the omitted baseline brand (Chefman), all the brands have higher market share after controlling for price, ratings, and product features.

Dash has the smallest coefficient of 0.18, so it barely outperforms the baseline.

In [51]:
coef_table[coef_table['feature'].str.startswith('year_')].sort_values('coefficient', ascending=False)

,feature,coefficient
16,year_2020,0.119071
17,year_2021,0.041900
19,year_2023,-0.003307
18,year_2022,-0.098860


The year with the largest dummy coefficient is 2020 (0.12), followed by 2021 (0.04).

Relative to the baseline year (2019), demand was slightly higher in 2020 and 2021. In contrast, 2023 and 2022 have negative coefficients, indicating demand was lower than the baseline in those years. This suggests the air fryer market peaked early in the pandemic era and softened by 2022–2023.

In [52]:
print('R-squared:', r2)

R-squared: 0.763453950091436


About 76.3% of the variation in log market share is explained by the model. This suggests a strong fit.